# shadowclock - render the REAL-geometry scenes on Kaggle (CPU, no GPU quota used)
**Settings:** Accelerator = **None** (CPU is enough), Internet = **On** (needed once, to download Python 3.11 and the Blender `bpy` module).

**Attach exactly two datasets**
1. `shadowclock-code`. It must contain these three files **exactly once each, in any folder layout** (the notebook copies them side by side): `generate_real_scenes.py`, `solar.py`, `verify_dataset.py`.
2. `shadowclock-osm`: the OpenStreetMap files you downloaded on your laptop (`data_real\osm_cache\*.json`, e.g. `delhi.json`, `london.json`, ...).

The notebook builds its own Python 3.11 environment with `bpy` (Kaggle's default Python is newer than the version the `bpy` wheel supports), renders every city whose OSM file is present, verifies the result and zips it.

In [ ]:
# ---- config ----
SAMPLES = 32          # Cycles samples per pixel
SIZE, GSD = 512, 0.5  # image size (px) and metres per pixel
DATES = "2027-03-20,2027-06-21,2027-12-21"
STEP_MIN = 60
DEFAULT_CITIES = ["delhi", "london", "newyork", "tokyo", "madrid", "chicago",
                  "sydney", "melbourne", "santiago", "buenosaires", "capetown", "johannesburg"]

In [ ]:
# ---- 1. Python 3.11 + bpy (takes a few minutes the first time) ----
import json, os, shlex, subprocess, sys, time
from pathlib import Path

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
VENV = WORK / "py311"

def run(cmd, tail=1200):
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    print((r.stdout + r.stderr)[-tail:])
    if r.returncode:
        raise RuntimeError(f"command failed (exit {r.returncode}): {' '.join(str(c) for c in cmd)}")

try:
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], tail=300)
except RuntimeError:
    run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", "uv"], tail=300)
if not (VENV / "bin" / "python").exists():
    run([sys.executable, "-m", "uv", "python", "install", "3.11"], tail=400)
    run([sys.executable, "-m", "uv", "venv", "--python", "3.11", VENV], tail=400)
    run([sys.executable, "-m", "uv", "pip", "install", "--python", VENV / "bin" / "python", "bpy", "numpy"], tail=600)
PY = VENV / "bin" / "python"
run([PY, "-c", "import bpy, numpy; print('bpy', bpy.app.version_string, '| numpy', numpy.__version__)"], tail=300)

In [ ]:
# ---- 2. find the code and the OSM files (strict: exactly one copy of each) ----
def unique(name):
    # Kaggle renames re-uploaded files to "name (4).py": accept "stem.ext" and "stem (n).ext", nothing else
    import re
    stem, ext = name.rsplit(".", 1)
    pat = re.compile(rf"^{re.escape(stem)}( \(\d+\))?\.{re.escape(ext)}$")
    hits = sorted(p for p in INPUT.rglob("*") if p.is_file() and pat.match(p.name))
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly ONE {name} under {INPUT}, found {len(hits)}: {[str(h) for h in hits]}. "
                           "Attach the right datasets and remove duplicates in the Input panel.")
    return hits[0]

import shutil
SRC = {f: unique(f) for f in ("generate_real_scenes.py", "solar.py", "verify_dataset.py")}
CODE = WORK / "code"                      # the three files may sit in different dataset folders: copy them side by side
CODE.mkdir(parents=True, exist_ok=True)
for f, p in SRC.items():
    shutil.copy(p, CODE / f)
print("copied into", CODE, "from:", {f: str(p.parent) for f, p in SRC.items()})
_txt = (CODE / "generate_real_scenes.py").read_text()
if "import generate_dataset" in _txt or "renderer helpers (self-contained)" not in _txt:
    raise RuntimeError("The generate_real_scenes.py in your Kaggle dataset is an OLD version (it still imports generate_dataset). "
                       "Delete it from the dataset, upload the newest generate_real_scenes.py, publish a new dataset version, "
                       "and make sure the notebook uses that version (Input panel -> dataset -> version).")
present = [c for c in DEFAULT_CITIES if list(INPUT.rglob(f"{c}.json"))]
missing = [c for c in DEFAULT_CITIES if c not in present]
assert present, "no <city>.json OSM files found: attach the dataset that contains data_real/osm_cache/*.json"
cache_dirs = {str(unique(f"{c}.json").parent) for c in present}
assert len(cache_dirs) == 1, f"the OSM files must all sit in one folder, found {cache_dirs}"
CACHE = Path(cache_dirs.pop())
print("code folder :", CODE)
print("OSM folder  :", CACHE)
print("cities found:", present)
print("cities MISSING (download them on the laptop and add them to the dataset):", missing or "none")

In [ ]:
# ---- 3. render (resumable: re-running skips images that already exist) ----
OUT = WORK / "data_real"
cmd = [PY, CODE / "generate_real_scenes.py", "--out", OUT, "--offline", "--osm-cache", CACHE,
       "--cities", ",".join(present), "--size", SIZE, "--gsd", GSD, "--dates", DATES, "--step-min", STEP_MIN,
       "--samples", SAMPLES, "--denoise", "--device", "cpu"]
t0 = time.time()
p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
tail = []
for line in p.stdout:
    tail.append(line)
    if line.startswith("[real]") or "WARNING" in line or "ERROR" in line:
        print(line.rstrip(), flush=True)
p.wait()
print(f"finished with exit code {p.returncode} after {(time.time() - t0) / 60:.1f} min")
if p.returncode:
    print("".join(tail[-15:]))
    raise RuntimeError("rendering did not finish cleanly: read the lines above, then re-run this cell (it resumes)")

In [ ]:
# ---- 4. verify, summarise, package ----
run([sys.executable, CODE / "verify_dataset.py", "--data", OUT], tail=800)
import zipfile
for p in sorted((OUT / "scenes").glob("scene_*.json")):
    info = json.load(open(p))
    print(f"{info['city']:13s} scene {info['scene']}: {info['n_buildings']:4d} buildings, explicit heights "
          f"{100 * info['explicit_height_fraction']:3.0f}%, tallest {info['max_height_m']:.0f} m, built-up {100 * info['built_fraction_of_field']:.0f}%")
zip_path = WORK / "data_real.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(OUT / "metadata.jsonl", "data_real/metadata.jsonl")
    for f in sorted((OUT / "images").glob("*.png")):
        z.write(f, "data_real/images/" + f.name)
    for f in sorted((OUT / "scenes").glob("*")):
        z.write(f, "data_real/scenes/" + f.name)
print(zip_path, f"{zip_path.stat().st_size / 1e6:.0f} MB;", len(list((OUT / "images").glob("*.png"))), "images")
print("Next: download data_real.zip from the Output panel, or use Save Version -> New Dataset from the notebook output.")